In [17]:
import numpy as np
import pandas as pd
import gc
import time
#from pathlib import Path #to keep importing datasets from relative paths consistent across OS


In [2]:
#current_directory = Path.cwd()
data_path = '../proj_data/ml-32m/'

links_data_path = data_path + 'links.csv'
movies_data_path = data_path + 'movies.csv'
ratings_data_path = data_path + 'ratings.csv'
tags_data_path = data_path + 'tags.csv'

#pull CSV files in

links_data = pd.read_csv(links_data_path) #infers whether it has columns or not
movies_data = pd.read_csv(movies_data_path) 
ratings_data = pd.read_csv(ratings_data_path) 
tags_data = pd.read_csv(tags_data_path) 

#slice the data to get a proportion



In [26]:
def slice_data(dataset, percent_slice):
    '''
    params: dataset [Full dataset], percent_slice[proportion (as an int NOT DECIMAL between 1 and 100 to slice]
    Take whatever % of the dataset and return the first x% of the data back
    
    '''
    
    row_ct = (percent_slice/100.0)*dataset.shape[0]
    return dataset[:int(row_ct)]

In [23]:
'''
links_slice = slice_data(links_data, 1)
movies_slice = slice_data(movies_data, 1)
ratings_slice = slice_data(ratings_data, 1)
tags_slice = slice_data(tags_data, 1)

print(f'Verifying shape of data. \nlinks data shape: {links_slice.shape}\nmovies_data shape: {movies_slice.shape} \nratings_data shape: {ratings_slice.shape} \ntags_data shape: {tags_slice.shape}')
'''


Verifying shape of data. 
links data shape: (875, 3)
movies_data shape: (875, 3) 
ratings_data shape: (320002, 4) 
tags_data shape: (20000, 4)


In [21]:

#get data shape info
print(f'Verifying shape of data. \nlinks data shape: {links_data.shape}\nmovies_data shape: {movies_data.shape} \nratings_data shape: {ratings_data.shape} \ntags_data shape: {tags_data.shape}')

#try joining data on keys. Start with only 10% of the data on each


Verifying shape of data. 
links data shape: (87585, 3)
movies_data shape: (87585, 3) 
ratings_data shape: (32000204, 4) 
tags_data shape: (2000072, 4)


In [23]:

def search_densest_sampled_matrix(dataset, sample_size, random_state):
    '''
    Args: 
        dataset (Pandas DataFrame): Dataframe that's going to be sampled-usually Ratings (needs to have movieId and userId columns)
        sample_size (int): size of the sample for the dataframe rows (usually 150,000)
        random_state(int): integer to reproduce the random sample
    returns:
        tuple of (sample count, random state for optimal matrix, sparsity of dataset in optimal matrix, movie_ct in optimal matrix, user_ct in optimal_matrix, dataset)
    '''
    best_combo = {
    'sample':sample_size,
    'random':random_state,
    'sparse':0.0,
    'movies':0,
    'users':0
    }

    start_time = time.time()
    
    for rando in np.arange(random_state,random_state+100):

        sample_ratings = dataset.sample(n=sample_size, replace=False, random_state=rando, axis = 0)

        unique_movies = sample_ratings['movieId'].nunique()
        unique_users = sample_ratings['userId'].nunique()
        sparsity_decimal = sample_size/(unique_movies * unique_users)
    
        if sparsity_decimal > best_combo['sparse']:
            best_combo['sample'] = sample_size
            best_combo['random'] = rando
            best_combo['sparse'] = sparsity_decimal
            best_combo['movies'] = unique_movies
            best_combo['users'] = unique_users

        del sample_ratings #this is a memory intensive process to clear memory after every call
        gc.collect() #clear memory space using garbage collector

    stop_time = time.time()
    print(f'Time elapsed for search was {(stop_time - start_time)/60: .4f} minutes')
    print(f'The best random state was {best_combo["random"]} with shape {best_combo["users"]} x {best_combo["movies"]} and a density score of {best_combo["sparse"]*100:.4f}%') 
    optimal_matrix_sample = dataset.sample(n=best_combo['sample'], replace=False, random_state=best_combo['random'], axis = 0)

    return (best_combo['sample'], best_combo['random'], best_combo['sparse'], best_combo['movies'], best_combo['users'], optimal_matrix_sample)



In [24]:
optimal_sample_ct, optimal_random_state, optimal_density, optimal_movie_ct, optimal_user_ct, optimal_ratings_DF = search_densest_sampled_matrix(ratings_data, 150000, 201)


Time elapsed for search was  3.0073 minutes
The best random state was 233 with shape 75816 x 13032 and a density score of 0.0152%


In [26]:
movies = pd.merge(pd.merge(pd.merge(optimal_ratings_DF, tags_data, on='movieId'), movies_data, on='movieId'), links_data, on='movieId')
print(f'movies DF shape is now: {movies.shape}')
movies.head()

movies DF shape is now: (103513795, 11)


,userId_x,movieId,rating,timestamp_x,userId_y,tag,timestamp_y,title,genres,imdbId,tmdbId
0,98121,1353,4.5,1468206256,7424,In Netflix queue,1312849218,"Mirror Has Two Faces, The (1996)",Comedy|Drama|Romance,117057,25189.0
1,98121,1353,4.5,1468206256,37008,personals ads,1137202938,"Mirror Has Two Faces, The (1996)",Comedy|Drama|Romance,117057,25189.0
2,98121,1353,4.5,1468206256,78213,columbia university,1528509830,"Mirror Has Two Faces, The (1996)",Comedy|Drama|Romance,117057,25189.0
3,98121,1353,4.5,1468206256,78213,professor,1528509830,"Mirror Has Two Faces, The (1996)",Comedy|Drama|Romance,117057,25189.0
4,98121,1353,4.5,1468206256,78213,sex,1528509830,"Mirror Has Two Faces, The (1996)",Comedy|Drama|Romance,117057,25189.0
